In [1]:
from datasets import load_dataset

ds = load_dataset("TAAC2026/data_sample_1000")
print(ds)

# 将 Dataset 转换为 Pandas 格式
df = ds['train'].to_pandas()

# 显示前 5 行
df.head()

README.md: 0.00B [00:00, ?B/s]

sample_data.parquet:   0%|          | 0.00/71.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['item_id', 'item_feature', 'label', 'seq_feature', 'timestamp', 'user_feature', 'user_id'],
        num_rows: 1000
    })
})


,item_id,item_feature,label,seq_feature,timestamp,user_feature,user_id
0,1548,"[{'feature_id': 6, 'feature_value_type': 'int_...","[{'action_time': 1770565361, 'action_type': 1}]","{'action_seq': [{'feature_id': 19, 'feature_va...",1770565208,"[{'feature_id': 65, 'feature_value_type': 'int...",user_3059
1,2537,"[{'feature_id': 6, 'feature_value_type': 'int_...","[{'action_time': 1770692680, 'action_type': 1}]","{'action_seq': [{'feature_id': 19, 'feature_va...",1770691713,"[{'feature_id': 58, 'feature_value_type': 'int...",user_3646
2,3244,"[{'feature_id': 6, 'feature_value_type': 'int_...","[{'action_time': 1770698500, 'action_type': 1}]","{'action_seq': [{'feature_id': 19, 'feature_va...",1770698286,"[{'feature_id': 55, 'feature_value_type': 'int...",user_2282
3,2939,"[{'feature_id': 6, 'feature_value_type': 'int_...","[{'action_time': 1770693229, 'action_type': 1}]","{'action_seq': [{'feature_id': 19, 'feature_va...",1770692106,"[{'feature_id': 56, 'feature_value_type': 'int...",user_1510
4,4054,"[{'feature_id': 6, 'feature_value_type': 'int_...","[{'action_time': 1770698669, 'action_type': 1}]","{'action_seq': [{'feature_id': 19, 'feature_va...",1770698383,"[{'feature_id': 56, 'feature_value_type': 'int...",user_3246


In [2]:
# 假设 label 列表不为空即代表有行为发生（点击/转化）
# 实际比赛中需根据 action_type 细分，这里先做一个基础提取
df['target'] = df['label'].apply(lambda x: 1 if len(x) > 0 else 0)

print("正负样本分布：")
print(df['target'].value_counts())

正负样本分布：
target
1    1000
Name: count, dtype: int64


In [3]:
# 取第一行数据
sample = df.iloc[0]['seq_feature']
# 检查 action_seq 里的第 19 号特征长度，是否等于 item_seq 里的第 29 号特征长度
print(len(sample['action_seq'][0]['int_array']) == len(sample['item_seq'][0]['int_array']))

False


In [5]:
import pandas as pd
import numpy as np

# ==========================================
# 步骤 1: 提取目标标签 (Target Label)
# ==========================================
# 原始数据中 label 是个列表，通常第一个元素的 action_type 就是我们要预测的行为
# 我们把它提取出来：1 代表有行为发生（点击等），0 代表没有
def parse_label(label_list):
    if len(label_list) > 0:
        return label_list[0]['action_type']
    return 0

df['target'] = df['label'].apply(parse_label)
print(f"标签解析完成，正样本数量: {df['target'].sum()}")


# ==========================================
# 步骤 2: 解析静态特征 (User/Item Features)
# ==========================================
# user_feature 和 item_feature 是 [{feature_id: 6, int_value: 96}, ...] 这种结构
# 我们写一个通用函数，提取出我们感兴趣的 feature_id 的数值
def get_static_feature(feature_list, target_id):
    for feat in feature_list:
        if feat['feature_id'] == target_id:
            # 优先取整数值，如果没有则取浮点数，再没有就取数组的第一个（如果有的话）
            if feat['int_value'] is not None: return feat['int_value']
            if feat['float_value'] is not None: return feat['float_value']
    return 0 # 默认填充 0

# 示例：提取几个之前分析表中比较重要的特征
df['item_industry'] = df['item_feature'].apply(lambda x: get_static_feature(x, 6)) # 假设 6 是行业
df['user_age_level'] = df['user_feature'].apply(lambda x: get_static_feature(x, 65)) # 假设 65 是年龄层

print("静态特征提取完成。")


# ==========================================
# 步骤 3: 解析序列特征 (Sequence Features) - 为 Interformer 准备
# ==========================================
# seq_feature 包含 action_seq, item_seq, content_seq
# 结构较深，需要多层提取。核心是拿到那个 int_array
def get_seq_data(seq_dict, group_name, target_id):
    """
    seq_dict: 原始的 seq_feature 列
    group_name: 'action_seq', 'item_seq' 或 'content_seq'
    target_id: 序列对应的 feature_id (如 19, 20 等)
    """
    try:
        group = seq_dict.get(group_name, [])
        for feat in group:
            if feat['feature_id'] == target_id:
                return feat['int_array']
    except:
        return []
    return []

# 提取行为类型序列 (ID 19) 和 对应的目标 ID 序列 (ID 20)
df['hist_action_type'] = df['seq_feature'].apply(lambda x: get_seq_data(x, 'action_seq', 19))
df['hist_item_ids'] = df['seq_feature'].apply(lambda x: get_seq_data(x, 'action_seq', 20))

print("用户行为序列提取完成。")


# ==========================================
# 修正后的 步骤 4: 序列补齐 (Padding)
# ==========================================
MAX_SEQ_LEN = 50

def pad_sequence(seq, max_len):
    # 【核心修正】：强制转为 list。
    # 这样 "+" 号就会执行“拼接”而不是“数学加法”
    curr_seq = list(seq) if seq is not None else []
    
    if len(curr_seq) >= max_len:
        # 截断：只取前 max_len 个
        return curr_seq[:max_len] 
    else:
        # 补齐：在后面接上一串 0
        return curr_seq + [0] * (max_len - len(curr_seq))

# 重新运行这一行
df['hist_item_ids_padded'] = df['hist_item_ids'].apply(lambda x: pad_sequence(x, MAX_SEQ_LEN))

print(f"修正后的序列补齐完成，固定长度为: {MAX_SEQ_LEN}")

# 看看解析后的最终数据预览
print("\n解析后的数据预览（前5行）：")
print(df[['user_id', 'item_id', 'target', 'hist_item_ids_padded']].head())

标签解析完成，正样本数量: 1103
静态特征提取完成。
用户行为序列提取完成。
修正后的序列补齐完成，固定长度为: 50

解析后的数据预览（前5行）：
     user_id  item_id  target  \
0  user_3059     1548       1   
1  user_3646     2537       1   
2  user_2282     3244       1   
3  user_1510     2939       1   
4  user_3246     4054       1   

                                hist_item_ids_padded  
0  [252, 201, 136, 32, 171, 252, 32, 15, 14, 0, 8...  
1  [126, 20, 134, 204, 204, 252, 201, 204, 125, 2...  
2  [136, 41, 14, 16, 160, 123, 16, 135, 16, 16, 5...  
3  [126, 32, 126, 126, 10, 5, 252, 227, 126, 126,...  
4  [548, 548, 548, 548, 0, 548, 5, 548, 21, 548, ...  


In [6]:
# 这一行如果能跑通，说明你的数据已经可以直接喂给深度学习模型了！
final_matrix = np.array(df['hist_item_ids_padded'].tolist())
print(f"最终矩阵形状: {final_matrix.shape}") 
# 预期输出应该是 (1000, 50)

最终矩阵形状: (1000, 50)


In [7]:
# 统计各个特征的最大 ID，用于确定 Embedding 矩阵的大小
# 注意：通常要加 1（因为 ID 从 0 开始）或加一个预留空间
item_vocab_size = df['item_id'].max() + 1
user_vocab_size = 10000 # 示例数据 user_id 是字符串，实际比赛需要转成数字 ID
# 序列中出现过的最大 ID
seq_item_vocab_size = final_matrix.max() + 1

print(f"目标物品词典大小: {item_vocab_size}")
print(f"用户历史序列词典大小: {seq_item_vocab_size}")

目标物品词典大小: 4068
用户历史序列词典大小: 563


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleInterformer(nn.Module):
    def __init__(self, item_vocab, seq_vocab, embed_dim=32):
        super(SimpleInterformer, self).__init__()
        
        # 1. Embedding 层：把数字 ID 变成 32 维的向量
        self.item_embed = nn.Embedding(item_vocab, embed_dim)
        self.seq_embed = nn.Embedding(seq_vocab, embed_dim)
        
        # 2. 注意力层 (Attention)：Interformer 的核心逻辑
        # 计算当前广告与历史序列的交互
        self.attention = nn.Linear(embed_dim * 2, 1)
        
        # 3. 全连接层 (MLP)：最后判断点不点的逻辑
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid() # 输出 0-1 之间的概率
        )

    def forward(self, target_item_id, history_seq_ids):
        # target_item_id: [batch_size]
        # history_seq_ids: [batch_size, 50]
        
        # 查表得到向量
        target_emb = self.item_embed(target_item_id) # [batch_size, embed_dim]
        seq_embs = self.seq_embed(history_seq_ids)   # [batch_size, 50, embed_dim]
        
        # --- 简化的注意力机制 (Interformer 思想) ---
        # 我们让 target_emb 复制 50 份，去跟序列里的每一个位置做对比
        target_emb_tile = target_emb.unsqueeze(1).expand(-1, 50, -1)
        combined = torch.cat([target_emb_tile, seq_embs], dim=-1) # [batch_size, 50, 64]
        
        atten_scores = self.attention(combined) # [batch_size, 50, 1]
        atten_scores = F.softmax(atten_scores, dim=1) # 归一化权重
        
        # 加权求和得到“用户的兴趣表示”
        user_interest = torch.sum(atten_scores * seq_embs, dim=1) # [batch_size, embed_dim]
        
        # 拼接目标广告和用户兴趣，最后预测
        final_input = torch.cat([target_emb, user_interest], dim=-1)
        output = self.mlp(final_input)
        
        return output.squeeze()

# 实例化模型
model = SimpleInterformer(item_vocab_size, seq_item_vocab_size)
print("\n模型架构已搭建完成：")
print(model)


模型架构已搭建完成：
SimpleInterformer(
  (item_embed): Embedding(4068, 32)
  (seq_embed): Embedding(563, 32)
  (attention): Linear(in_features=64, out_features=1, bias=True)
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
    (3): Sigmoid()
  )
)


In [9]:
import torch.optim as optim

# ==========================================
# 步骤 1: 准备 Tensor 数据
# ==========================================
# 我们需要把 Pandas 的数据转成 PyTorch 能认的张量
# 注意：ID类特征用 LongTensor，标签用 FloatTensor
X_target_item = torch.LongTensor(df['item_id'].values)
X_history_seq = torch.LongTensor(final_matrix) # 之前解析好的 (1000, 50)
Y_label = torch.FloatTensor(df['target'].values)

# ==========================================
# 步骤 2: 定义“教练”（损失函数和优化器）
# ==========================================
# BCELoss: 二分类交叉熵损失，专门用于“点不点”这种 0/1 预测
criterion = nn.BCELoss() 

# Adam: 目前工业界最常用的优化器，能自动调整学习率
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ==========================================
# 步骤 3: 开始训练循环 (Epochs)
# ==========================================
print("开始训练...")
model.train() # 切换到训练模式

for epoch in range(10): # 暂时跑 10 轮看看
    # 1. 清空之前的梯度
    optimizer.zero_grad()
    
    # 2. 前向传播：把数据喂给模型，得到预测概率
    outputs = model(X_target_item, X_history_seq)
    
    # 3. 计算损失：预测值和真实 label 差了多少？
    loss = criterion(outputs, Y_label)
    
    # 4. 反向传播：把“误差”传回给每个神经元
    loss.backward()
    
    # 5. 更新参数：让神经元根据误差微调自己的权重
    optimizer.step()
    
    if (epoch + 1) % 2 == 0:
        print(f"Epoch [{epoch+1}/10], Loss: {loss.item():.4f}")

print("\n初步训练尝试完成！")

开始训练...


RuntimeError: all elements of target should be between 0 and 1

In [10]:
# ==========================================
# 1. 彻底修正标签：把 1, 2, 4 等全部统一成 1.0 (代表点击或发生行为)
# ==========================================
def binary_label_parse(label_list):
    # 只要列表不为空，就代表发生了某种正向行为，记为 1.0
    if isinstance(label_list, (list, np.ndarray)) and len(label_list) > 0:
        return 1.0 
    return 0.0

# 重新生成 target 列，确保只有 0.0 和 1.0
df['target'] = df['label'].apply(binary_label_parse)

# 打印一下，看看现在是不是只有 0 和 1 了
print("修正后的标签分布：")
print(df['target'].value_counts())

# ==========================================
# 2. 重新准备张量 (Tensor)
# ==========================================
X_target_item = torch.LongTensor(df['item_id'].values)
X_history_seq = torch.LongTensor(final_matrix)
# 这一步非常关键：必须是 FloatTensor，且数值只能是 0.0 或 1.0
Y_label = torch.FloatTensor(df['target'].values)

# ==========================================
# 3. 重新启动训练循环
# ==========================================
print("\n开始重新训练...")
model.train()
# 重新定义优化器，确保它能从头开始学习
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.BCELoss()

for epoch in range(10):
    optimizer.zero_grad()
    outputs = model(X_target_item, X_history_seq)
    
    # 这里的 outputs 是模型预测的概率，Y_label 是我们刚才修正的 0/1 标签
    loss = criterion(outputs, Y_label)
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 2 == 0:
        print(f"Epoch [{epoch+1}/10], Loss: {loss.item():.4f}")

print("\n恭喜！修正后的训练已完成。")

修正后的标签分布：
target
1.0    1000
Name: count, dtype: int64

开始重新训练...
Epoch [2/10], Loss: 0.6684
Epoch [4/10], Loss: 0.6365
Epoch [6/10], Loss: 0.6057
Epoch [8/10], Loss: 0.5759
Epoch [10/10], Loss: 0.5469

恭喜！修正后的训练已完成。
